# Convert outputs to yearly zarr files

In [1]:
import re
import os
import sys

import zarr
import yaml
from glob import glob
from datetime import datetime, timedelta

import numpy as np
import xarray as xr

In [2]:
sys.path.insert(0, os.path.realpath('../libs/'))
import verif_utils as vu

### Get the target data for coord reference

In [3]:
# fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2020.zarr'
# ds_target = xr.open_zarr(fn_target)

## Gather prognostic outputs

### Combine raw netCDF4 to zarr

In [4]:
# ind_start = 0
# ind_end = 24*366 - 1

In [12]:
source_dir = '/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_B1H/'

fn_all = vu.get_nc_files(source_dir)[1]
fn_all = sorted(fn_all, key=lambda x: int(re.search(r'_(\d+)\.nc$', x).group(1)))[:-1]

In [13]:
fn_all[-1]

'/glade/derecho/scratch/ksha/DWC/RAW_OUTPUT/CONUS_GP_B1H/2023-01-01T00Z/pred_2023-01-01T00Z_17641.nc'

In [8]:
# for fn in fn_all[ind_start:ind_end]:
#     ds = xr.open_dataset(fn)
#     ds_collect.append(ds)

# ds_final = xr.concat(ds_collect, dim='time')

# ds_final = ds_final.rename({'latitude': 'south_north', 'longitude': 'west_east', 'level': 'bottom_top'})
# ds_final['west_east'] = np.arange(336).astype(np.float32)
# ds_final['south_north'] = np.arange(336).astype(np.float32)
# ds_final['bottom_top'] = np.arange(12).astype(np.float32)

# # =================================================== #
# # combine with diag
# # ds_target = ds_target.sel(time=ds_final['time'])
# # ds_final = xr.merge([ds_final, ds_target[['WRF_precip_025', 'WRF_radar_composite_025', 'WRF_OLR', 'WRF_TCC', 'WRF_GLW', 'WRF_SWDOWN', 'WRF_SMOIS', 'WRF_TSLB']]])
# ds_final = ds_final.chunk({'time': 12, 'bottom_top': 12, 'south_north': 336, 'west_east': 336})

# # =================================================== #
# # zarr encodings
# dict_encoding = {}
# varnames = list(ds_final.keys())
# varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']

# chunk_size_3d = dict(chunks=(12, 336, 336))
# chunk_size_4d = dict(chunks=(12, 12, 336, 336))
# compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)

# for i_var, var in enumerate(varnames):
#     if var in varname_4D:
#         dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
#     else:
#         dict_encoding[var] = {'compressor': compress, **chunk_size_3d}

# save_name = f'/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_single_clean_{ind_start:04d}_{ind_end:04d}_2020-01-01T00Z.zarr'
# # ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)

### Concat zarr to yearly

In [4]:
def extract_start_index(path):
    # Find numbers like _0000_1000_ and take the first one
    match = re.search(r'_(\d+)_\d+_', path)
    return int(match.group(1)) if match else float('inf')

def year_from_dt64(dt64):
    """Convert numpy.datetime64[ns] to integer year."""
    return dt64.astype("datetime64[Y]").astype(int) + 1970

def first_day_of_year(dt64):
    """Return the datetime64[ns] of the first day of the year for dt64."""
    return dt64.astype("datetime64[Y]").astype("datetime64[ns]")

### B3H

In [18]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B3H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H/opt_B3H_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B3H_2023.zarr


In [ ]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B3H_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B3H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

### B6H

In [19]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B6H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H/opt_B6H_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_B6H_2023.zarr


In [ ]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/B6H_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
    
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_B6H_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

### GDAS

In [6]:
for year in range(2020, 2024):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
        
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_GDAS_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2020.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_9000_10000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2021.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_18000_19000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2022.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS/opt_GDAS_27000_28000_2020-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2023.zarr


In [5]:
for year in range(2024, 2025):
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS_p2/'
    fn_all = sorted(glob(base_dir + '*.zarr'), key=extract_start_index)
    
    ds_collect = []
    flag_add_previous = True
    
    for i_fn, fn in enumerate(fn_all):
        ds = xr.open_zarr(fn)
        
        # Extract first timestamp
        first_day = ds['time'].values[0]
        year_ = year_from_dt64(first_day)
    
        if year_ != year:
            continue  # Skip years not matching target
        
        # If file starts mid-year, add the previous file once
        if (first_day != first_day_of_year(first_day)) and (i_fn > 0) and flag_add_previous and (year != 2020):
            print(fn)
            ds_prev = xr.open_zarr(fn_all[i_fn - 1])
            ds_collect.append(ds_prev)
            flag_add_previous = False
    
        ds_collect.append(ds)
    
    # Combine and trim to exact calendar year
    if not ds_collect:
        raise ValueError(f"No data found for year {year}")
    
    ds_final = xr.concat(ds_collect, dim='time')
    
    ds_final = ds_final.sel(time=slice(f"{year}-01-01T00:00:00", f"{year}-12-31T23:00:00"))
    ds_final = ds_final.chunk({'time': 12, 'south_north': 336, 'west_east': 336, 'bottom_top': 12})
    
    # =================================================== #
    # zarr encodings
    dict_encoding = {}
    varnames = list(ds_final.keys())
    varname_4D = ['WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05', 'WRF_P']
    
    chunk_size_3d = dict(chunks=(12, 336, 336))
    chunk_size_4d = dict(chunks=(12, 12, 336, 336))
    compress = zarr.Blosc(cname='zstd', clevel=1, shuffle=zarr.Blosc.SHUFFLE, blocksize=0)
    
    for i_var, var in enumerate(varnames):
        if var in varname_4D:
            dict_encoding[var] = {'compressor': compress, **chunk_size_4d}
        else:
            dict_encoding[var] = {'compressor': compress, **chunk_size_3d}
    
    base_dir = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/'
    save_name = base_dir + f'opt_GDAS_{year}.zarr'
    ds_final.to_zarr(save_name, mode='w', consolidated=True, compute=True, encoding=dict_encoding)
    print(save_name)

/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/GDAS_p2/opt_GDAS_9000_10000_2023-01-01T00Z.zarr
/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_GDAS_2024.zarr


### Check NaNs

In [29]:
def check_nans_ds(ds):
    nan_vars = []
    for var in ds.data_vars:
        # Check if there are any NaNs in the variable
        if ds[var].isnull().any():
            nan_vars.append(var)
    return nan_vars

In [30]:
# ds_temp = ds_final
# nan_vars = check_nans_ds(ds_temp)

# if nan_vars:
#     print('Dataset contains NaNs in the following variables:')
#     for var in nan_vars:
#         print(f"- {nan_vars}")
#     print(f"File: {fn}")
# else:
#     print('Dataset does not contain NaNs')

### Add 2021 for diag model

In [4]:
fn_target = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/C404/C404_GP_2021.zarr'
ds_target = xr.open_zarr(fn_target)
ds_save = ds_target.isel(time=slice(120))
save_name = '/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/opt_init_ERA5/prog_outputs/opt_m3_clean_2021-01-01T00Z.zarr'
# ds_save.to_zarr(save_name, mode='w', consolidated=True, compute=True)